# T23 / E04 — Chunk-aware chia theo cửa sổ token

E03 đã cho thấy hình dạng phân bố chú ý mang tín hiệu: macro-F1 **0,7567** so với **0,7451** của
E02, và ECE giảm hơn một nửa. Nhưng E03 mới là **một** cách chia đoạn. Notebook này quét cách
chia thứ hai — cửa sổ token cố định — ở ba cỡ 64, 128 và 256, theo mục E05 của
`docs/EXPERIMENTS.md`. T24 chốt một cái và giữ nguyên cho mọi thí nghiệm sau.

## Đã biết trước khi chạy: cửa sổ 256 gần như chắc chắn hỏng

Đo trên CPU bằng `scripts/probe_chunking.py`, **không tốn giây GPU nào**. Ngữ cảnh ViHallu chỉ
dài trung vị **218 token**:

| cách chia | TB số đoạn | trung vị | chỉ 1 đoạn | ≤ 2 đoạn |
|---|---|---|---|---|
| câu, min_words=5 (E03) | 5,3 | 5 | 1,1 % | 5,8 % |
| cửa sổ 64, bước 32 | 7,1 | 6 | 0,0 % | 0,0 % |
| cửa sổ 128, bước 64 | 3,3 | 3 | 0,1 % | 34,5 % |
| **cửa sổ 256, bước 128** | **1,4** | **1** | **67,2 %** | **92,7 %** |

Với **một** đoạn, năm đặc trưng hình dạng thành hằng số `(0, 1, 0, 1, 0)` bất kể mô hình làm gì.
Nên ở 67 % dữ liệu, cửa sổ 256 không nói được gì hơn E02.

**Vẫn chạy nó.** Bảng 3 phải được điền bằng **số đo** chứ không bằng suy luận, và biết trước
hình dạng của kết quả là một cách kiểm tra rằng đường ống chạy đúng: nếu cửa sổ 256 **không**
rơi về gần E02 thì có gì đó sai, chứ không phải có phát hiện mới.

## Ba lượt trích riêng, không dùng chung được

Ranh giới đoạn nằm trong `extraction_hash`, nên đổi cỡ cửa sổ là đổi mảng theo đoạn — phải chạy
lại mô hình đọc. Ba lượt, mỗi lượt train và dev, khoảng **3,2 giờ GPU**.

## Không chấm tập test ở T23

Mọi ô chấm điểm đều chạy `--dev-only`. T23 chỉ cần cột `macro-F1 dev` của Bảng 3; T24 chọn xong
mới trích tập test cho **đúng cấu hình thắng** và chấm một lần. Nhờ vậy "chọn trên dev" là một
sự thật về thứ đã được tính, chứ không phải một lời hứa về thứ đã không nhìn.

## Chuẩn bị

Hai ô dưới đây giống notebook T22. Khoảng 2 phút.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: a2da072 T22: lưu notebook đã chạy làm minh chứng (#53)


In [2]:
# Ô 2 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
# hình đọc 7B. Không có nó thì mô hình phải nạp ở float16 và tràn 16 GB.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate bitsandbytes

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.
vihallulens 0.1.0 requires rank-bm25, which is not instal

In [3]:
# Ô 3 — chuẩn bị dữ liệu và kiểm tra môi trường. Khoảng 1 phút, chạy CPU.
# test_chunking_config.py là bộ kiểm thử viết riêng cho T23: nó khẳng định tokenizer và cỡ cửa
# sổ thật sự tới được bộ chia đoạn. Trước T23 thì KHÔNG — và lỗi đó chỉ lộ ra ở mẫu đầu tiên
# của lượt GPU, sau khi đã nạp xong mô hình.
get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset vihallu")
get_ipython().system("python scripts/split_data.py --only vihallu")
get_ipython().system(
    "python -m pytest tests/test_chunking_config.py tests/test_chunk_features.py -q"
)


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : a2da072 T22: lưu notebook đã chạy làm minh chứng (#53)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.16.1
  bitsandbytes     : 0.50.2
  accelerate       : 1.14.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIHALLU
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 7,000
  ngữ cảnh duy nhất     : 3,

In [4]:
# Ô 4 — đếm số đoạn thật sự trên dữ liệu thật. Khoảng 1 phút, chạy CPU.
# Đây là bảng đã dán ở đầu notebook. Chạy lại để xác nhận số trên Kaggle khớp số đã đo ở nhà.
get_ipython().system("python scripts/probe_chunking.py --dataset vihallu")

config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 3.31MB/s]
tokenizer_config.json: 7.30kB [00:00, 7.63MB/s]
vocab.json: 2.78MB [00:00, 48.4MB/s]
merges.txt: 1.67MB [00:00, 120MB/s]
tokenizer.json: 7.03MB [00:00, 135MB/s]

T23 — SỐ ĐOẠN THEO TỪNG CÁCH CHIA · vihallu · 7,000 ngữ cảnh
  độ dài ngữ cảnh (token, Qwen/Qwen2.5-7B-Instruct): trung bình 243, trung vị 218, p95 423, lớn nhất 2,347

  cách chia                    TB  trung vị    p95    max    1 đoạn   ≤2 đoạn
  ---------------------------------------------------------------------------
  câu, min_words=5            5.3         5      9     42     1.1%      5.8%
  cửa sổ 64, bước 32          7.1         6     13     73     0.0%      0.0%
  cửa sổ 128, bước 64         3.3         3      6     36     0.1%     34.5%
  cửa sổ 256, bước 128        1.4         1      3     18    67.2%     92.7%

  Cột '1 đoạn' là phần dữ liệu mà năm đặc trưng hình dạng thành hằng số
  (0, 1, 0, 1, 0) — ở đó chunk-aware không nói được

## Trích đặc trưng — ba cỡ cửa sổ, mỗi cỡ hai tập

Sáu ô, khoảng **3,2 giờ**. Mỗi ô **chạy lại được**: mẫu nào tính xong ghi xuống ngay, chạy lại
thì bỏ qua phần đã có. Mất phiên giữa chừng thì chạy lại đúng ô đó, không mất phần đã trả tiền.

**Đọc gì trong lúc chạy:** cột `lỗi` phải là 0, và dòng `chia đoạn` phải in đúng cỡ cửa sổ của
ô đó — `token_window, cửa sổ 64 bước 32` chẳng hạn. Nếu nó in `sentence` thì đang chạy nhầm
cấu hình và sẽ ghi đè lên lượt trích của E03.

In [5]:
# Ô 5 — cửa sổ 64, tập train. Khoảng 57 phút.
!python scripts/extract_features.py --config configs/e04_chunk_window64_vihallu.yaml --split train


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e04_chunk_window64_vihallu.yaml  (hash 3d1f6204fc98)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : token_window, cửa sổ 64 bước 32
  bộ dữ liệu            : vihallu/train, 5,600 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  đã có sẵn             : 0 mẫu trong vihallu_train_3d1f6204fc98.jsonl
  còn phải chạy         : 5,600 mẫu
model.safetensors.index.json: 27.8kB [00:00, 58.3MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:04<00:00, 16.10s/it]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:04<00:00, 301MB/s]
Loading weights:   0%|                                  | 0/339 [00:00<?, ?it/s]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:09<00:00, 301MB/s]
L

In [6]:
# Ô 6 — cửa sổ 64, tập dev. Khoảng 7 phút. Tập này để CHỌN cách gộp đầu, không để báo kết quả.
!python scripts/extract_features.py --config configs/e04_chunk_window64_vihallu.yaml --split dev


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e04_chunk_window64_vihallu.yaml  (hash 3d1f6204fc98)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : token_window, cửa sổ 64 bước 32
  bộ dữ liệu            : vihallu/dev, 700 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  đã có sẵn             : 0 mẫu trong vihallu_dev_3d1f6204fc98.jsonl
  còn phải chạy         : 700 mẫu
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 23.44it/s]


--------------------------------------------------------------------------------
  đã ghi thêm           : 700 mẫu, tổng 700
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/700
  có lớp tràn số        : 0/700
  thời gian             : 6.3 phút, 541 ms/mẫu
  file                  : data/processed/

In [7]:
# Ô 7 — cửa sổ 128, tập train. Khoảng 57 phút.
!python scripts/extract_features.py --config configs/e04_chunk_window128_vihallu.yaml --split train


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e04_chunk_window128_vihallu.yaml  (hash d08935b76502)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : token_window, cửa sổ 128 bước 64
  bộ dữ liệu            : vihallu/train, 5,600 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  đã có sẵn             : 0 mẫu trong vihallu_train_d08935b76502.jsonl
  còn phải chạy         : 5,600 mẫu
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 23.63it/s]


--------------------------------------------------------------------------------
  đã ghi thêm           : 5,600 mẫu, tổng 5,600
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/5,600
  có lớp tràn số        : 0/5,600
  thời gian             : 50.5 phút, 541 ms/mẫu
  file                

In [8]:
# Ô 8 — cửa sổ 128, tập dev. Khoảng 7 phút.
!python scripts/extract_features.py --config configs/e04_chunk_window128_vihallu.yaml --split dev


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e04_chunk_window128_vihallu.yaml  (hash d08935b76502)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : token_window, cửa sổ 128 bước 64
  bộ dữ liệu            : vihallu/dev, 700 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  đã có sẵn             : 0 mẫu trong vihallu_dev_d08935b76502.jsonl
  còn phải chạy         : 700 mẫu
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 23.99it/s]


--------------------------------------------------------------------------------
  đã ghi thêm           : 700 mẫu, tổng 700
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/700
  có lớp tràn số        : 0/700
  thời gian             : 6.3 phút, 540 ms/mẫu
  file                  : data/processe

In [9]:
# Ô 9 — cửa sổ 256, tập train. Khoảng 57 phút.
# Cỡ này đã biết trước là gần như thoái hóa. Chạy để đo, không để kỳ vọng.
!python scripts/extract_features.py --config configs/e04_chunk_window256_vihallu.yaml --split train


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e04_chunk_window256_vihallu.yaml  (hash dd4642b78817)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : token_window, cửa sổ 256 bước 128
  bộ dữ liệu            : vihallu/train, 5,600 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  đã có sẵn             : 0 mẫu trong vihallu_train_dd4642b78817.jsonl
  còn phải chạy         : 5,600 mẫu
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 23.98it/s]


--------------------------------------------------------------------------------
  đã ghi thêm           : 5,600 mẫu, tổng 5,600
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/5,600
  có lớp tràn số        : 0/5,600
  thời gian             : 50.4 phút, 540 ms/mẫu
  file               

In [10]:
# Ô 10 — cửa sổ 256, tập dev. Khoảng 7 phút.
!python scripts/extract_features.py --config configs/e04_chunk_window256_vihallu.yaml --split dev


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e04_chunk_window256_vihallu.yaml  (hash dd4642b78817)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : token_window, cửa sổ 256 bước 128
  bộ dữ liệu            : vihallu/dev, 700 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  đã có sẵn             : 0 mẫu trong vihallu_dev_dd4642b78817.jsonl
  còn phải chạy         : 700 mẫu
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 23.99it/s]


--------------------------------------------------------------------------------
  đã ghi thêm           : 700 mẫu, tổng 700
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/700
  có lớp tràn số        : 0/700
  thời gian             : 6.3 phút, 540 ms/mẫu
  file                  : data/process

## Chấm trên dev

Ba ô, chạy CPU, mỗi ô vài phút. Copy toàn bộ output của cả ba.

Dòng cần lấy ra là `macro-F1 trên DEV` ở cuối mỗi ô — đó là một ô của Bảng 3. Và dòng
`ngữ cảnh chỉ 1 đoạn` ở đầu, vì nó là lời giải thích cho con số ấy.

In [11]:
# Ô 11 — cửa sổ 64, chọn cách gộp đầu trên dev. Không đụng tập test.
!python scripts/run_chunk_aware.py --config configs/e04_chunk_window64_vihallu.yaml --dev-only


E04_CHUNK_AWARE_WINDOW64_VIHALLU — BỘ PHÁT HIỆN CHUNK-AWARE
  cấu hình              : configs/e04_chunk_window64_vihallu.yaml  (trích 3d1f6204fc98)
  chia đoạn             : token_window, cửa sổ 64 bước 32
  nhóm đặc trưng        : basic, chunk_aware, stability
  khối                  : lookback_total, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  train / dev           : 5,600 / 700 mẫu
  chế độ                : --dev-only, KHÔNG chấm tập test
  lưới lớp × đầu        : 27 × 28
  số đoạn mỗi ngữ cảnh  : trung bình 7.1, trung vị 6, tối đa 33
  ngữ cảnh chỉ 1 đoạn   : 0/5,600 (0.0 %)   ← chunk-aware thoái hóa thành gộp ở đó
  đặc trưng nếu giữ hết : 4,536

--------------------------------------------------------------------------------
CHỌN CÁCH GỘP ĐẦU CHÚ Ý — chấm trên tập DEV, không đụng tập test
--------------------------------------------------------------------------------
  Cách gộp                           số chiều   macro-F1 dev
  all                 

In [12]:
# Ô 12 — cửa sổ 128, chọn cách gộp đầu trên dev. Không đụng tập test.
!python scripts/run_chunk_aware.py --config configs/e04_chunk_window128_vihallu.yaml --dev-only


E04_CHUNK_AWARE_WINDOW128_VIHALLU — BỘ PHÁT HIỆN CHUNK-AWARE
  cấu hình              : configs/e04_chunk_window128_vihallu.yaml  (trích d08935b76502)
  chia đoạn             : token_window, cửa sổ 128 bước 64
  nhóm đặc trưng        : basic, chunk_aware, stability
  khối                  : lookback_total, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  train / dev           : 5,600 / 700 mẫu
  chế độ                : --dev-only, KHÔNG chấm tập test
  lưới lớp × đầu        : 27 × 28
  số đoạn mỗi ngữ cảnh  : trung bình 3.3, trung vị 3, tối đa 16
  ngữ cảnh chỉ 1 đoạn   : 8/5,600 (0.1 %)   ← chunk-aware thoái hóa thành gộp ở đó
  đặc trưng nếu giữ hết : 4,536

--------------------------------------------------------------------------------
CHỌN CÁCH GỘP ĐẦU CHÚ Ý — chấm trên tập DEV, không đụng tập test
--------------------------------------------------------------------------------
  Cách gộp                           số chiều   macro-F1 dev
  all              

In [13]:
# Ô 13 — cửa sổ 256, chọn cách gộp đầu trên dev. Không đụng tập test.
!python scripts/run_chunk_aware.py --config configs/e04_chunk_window256_vihallu.yaml --dev-only


E04_CHUNK_AWARE_WINDOW256_VIHALLU — BỘ PHÁT HIỆN CHUNK-AWARE
  cấu hình              : configs/e04_chunk_window256_vihallu.yaml  (trích dd4642b78817)
  chia đoạn             : token_window, cửa sổ 256 bước 128
  nhóm đặc trưng        : basic, chunk_aware, stability
  khối                  : lookback_total, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  train / dev           : 5,600 / 700 mẫu
  chế độ                : --dev-only, KHÔNG chấm tập test
  lưới lớp × đầu        : 27 × 28
  số đoạn mỗi ngữ cảnh  : trung bình 1.4, trung vị 1, tối đa 8
  ngữ cảnh chỉ 1 đoạn   : 3,726/5,600 (66.5 %)   ← chunk-aware thoái hóa thành gộp ở đó
  đặc trưng nếu giữ hết : 4,536

--------------------------------------------------------------------------------
CHỌN CÁCH GỘP ĐẦU CHÚ Ý — chấm trên tập DEV, không đụng tập test
--------------------------------------------------------------------------------
  Cách gộp                           số chiều   macro-F1 dev
  all         

In [14]:
# Ô 14 — lấy đặc trưng thô về. Sáu file, dùng lại được ở T24 và E12.
import shutil
from pathlib import Path

for path in sorted(Path("data/processed").glob("*.jsonl")):
    shutil.copy(path, f"/kaggle/working/{path.name}")
    print(f"{path.name}  {path.stat().st_size / 1024**2:.1f} MB")

vihallu_dev_3d1f6204fc98.jsonl  35.2 MB
vihallu_dev_d08935b76502.jsonl  35.2 MB
vihallu_dev_dd4642b78817.jsonl  27.2 MB
vihallu_train_3d1f6204fc98.jsonl  281.9 MB
vihallu_train_d08935b76502.jsonl  281.8 MB
vihallu_train_dd4642b78817.jsonl  216.3 MB


## Đọc kết quả thế nào

**T23 không có mốc phải vượt.** Nó là một phép quét, và kết quả của nó là một bảng chứ không
phải một con số thắng thua. Đừng đọc "cửa sổ 64 thua E03" như một thất bại — E05 tồn tại chính
là để đo xem cách chia nào hợp dữ liệu này hơn.

Bốn thứ cần nhìn:

1. **`ngữ cảnh chỉ 1 đoạn`**, in ở đầu mỗi ô chấm điểm. Với cửa sổ 256 phải ra khoảng **67 %**.
   Khớp thì đường ống đúng. Không khớp thì bộ chia đoạn đang làm điều gì đó khác dự đoán, và
   phải dừng lại kiểm trước khi đọc bất kỳ điểm số nào.

2. **`macro-F1 trên DEV`** của cả ba, đặt cạnh **0,7768** — điểm dev của E03 chia theo câu. Đây
   là bốn ô của Bảng 3.

3. **Bảng chọn cách gộp đầu.** Nếu cả ba cỡ cửa sổ đều chọn cùng cách gộp mà E03 đã chọn
   (`topk_heads k=32`) thì lựa chọn ấy ổn định theo cách chia đoạn — đáng viết vào báo cáo. Nếu
   mỗi cỡ chọn một kiểu thì phải nói rõ rằng nó phụ thuộc cấu hình.

4. **Cửa sổ 64 so với 128.** Đây mới là phần thú vị: 64 cho 7,1 đoạn, 128 cho 3,3 đoạn, còn E03
   chia theo câu cho 5,3 đoạn. Nếu điểm tăng theo số đoạn thì thứ quyết định là **độ phân giải**
   chứ không phải ranh giới ngữ nghĩa của câu — đó là một phát hiện về cơ chế, không chỉ là một
   ô trong bảng.

## Một cảnh báo về cách đọc entropy ở cửa sổ chồng lấn

Bước bằng nửa cửa sổ nên các đoạn **chồng lấn**: một token trong vùng chồng được đếm cho cả hai
đoạn. Véc-tơ theo đoạn vì thế không còn là một phân bố theo nghĩa chặt, và token ở hai đầu ngữ
cảnh chỉ được phủ một lần trong khi token ở giữa được phủ hai lần.

Entropy và Gini vẫn tính được sau khi chuẩn hóa, nhưng phải đọc là *"chú ý tản trên các cửa
sổ"* chứ không phải *"tản trên các phần rời nhau của ngữ cảnh"*. Chia theo câu không có vấn đề
này vì các câu phủ kín và không đè lên nhau. Nếu cửa sổ thua câu ở T24 thì đây là một trong hai
cách giải thích, và cách kia là số đoạn.